# 🎙️ Train Virgo's own Khmer voice — Kaggle (free GPU)

This trains **Virgo-1.0-Angkor's own Khmer voice** from scratch (a VITS model) on **OpenSLR 42**, Google's high-quality Khmer speech recordings (CC BY-SA 4.0, commercial use allowed with attribution). No Meta MMS inside, so it's **safe for a paid product**.

**First time only**
1. Kaggle → **Settings → Phone verification** (needed for GPU and internet).
2. **Add-ons → Secrets**: `HF_TOKEN` (your Hugging Face token, with **write** access), ticked for this notebook.

**Each run**
1. **Session options**: **Accelerator → GPU T4 x2** (or P100), **Internet → On**.
2. **Save Version → Save & Run All (Commit)**. It runs in the background for about 11 hours, so you can close the tab or turn off your PC.
3. It saves to your private Hugging Face repo `Virgo-1.0-Angkor-Voice`. **Run it again and it continues where it stopped.**

**How long?** A clear voice needs roughly 100,000+ training steps, which takes **3–5 runs** (one or two weeks of free Kaggle GPU). Each run ends by speaking three test sentences, so you can hear it improve. As soon as one run has finished, the Virgo server notebooks use this voice automatically for Khmer.

**Your own recordings (optional):** upload a Kaggle dataset with `.wav` files and a `metadata.csv` (`file_name,sentence`), add it to this notebook, and set `EXTRA` below to its folder (e.g. `/kaggle/input/my-khmer-voice`). A single clear speaker with 1–2 hours of speech gives Virgo a unique voice.

In [ ]:
# ⚙️ Settings
VOICE_REPO = ""   # leave empty for <your Hugging Face name>/Virgo-1.0-Angkor-Voice
HOURS = 11        # stop, save and upload after this long (a Kaggle session is 12 hours)
EXTRA = ""        # optional: folder of your own recordings (.wav + metadata.csv)

In [ ]:
# 1️⃣ Set up
import os, subprocess, sys, torch
assert torch.cuda.is_available(), "❌ No GPU: Session options → Accelerator → GPU"
WORK = "/kaggle/working"
os.chdir(WORK)
if not os.path.isdir("virgo-models/.git"):
    r = subprocess.run(["git", "clone", "-q", "https://github.com/nangmrr752/virgo-models"])
    assert r.returncode == 0, "❌ Can't download: Session options → Internet → On (needs phone verification)"
os.chdir(f"{WORK}/virgo-models")
subprocess.run(["git", "pull", "-q"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "coqui-tts==0.27.2", "librosa", "huggingface_hub"], check=True)
from huggingface_hub import login, HfApi
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token = None
assert token, "❌ Add the HF_TOKEN secret (Add-ons → Secrets) and tick it for this notebook"
login(token=token)
REPO = VOICE_REPO or f"{HfApi().whoami()['name']}/Virgo-1.0-Angkor-Voice"
print("✅ Ready. The voice is saved to", REPO)

def run(cmd):
    """Runs a step and shows its output here; on failure, shows the last lines (send them to Claude)."""
    import subprocess, collections
    tail = collections.deque(maxlen=60)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        tail.append(line)
    if proc.wait() != 0:
        print("\n" + "=" * 60 + "\n❌ It stopped with an error. Send these last lines to Claude:\n" + "=" * 60)
        print("".join(tail))
        raise SystemExit(f"❌ {next((a for a in cmd if a.endswith('.py')), cmd[0])} failed (exit code {proc.returncode})")


In [ ]:
# 2️⃣ Train (continues the last run on Hugging Face, if there is one)
import subprocess, sys
cmd = [sys.executable, "-u", "speech/train_voice.py", "--push", REPO, "--hours", str(HOURS), "--work", "/kaggle/working/voice"]
if EXTRA:
    cmd += ["--extra", EXTRA]
run(cmd)

In [ ]:
# 3️⃣ Listen to Virgo's voice now
import os, sys
from IPython.display import Audio, display
os.environ["VIRGO_VOICE_PYTHON"] = sys.executable
sys.path.insert(0, "speech")
from khmer_voice import KhmerVoice
voice = KhmerVoice("speech/out/virgo-khmer-voice")
for text in ["សួស្តី ខ្ញុំឈ្មោះ Virgo ជាជំនួយការ AI របស់ KSN។", "តើអ្នកសុខសប្បាយជាទេ?", "អរគុណច្រើន។"]:
    wave, rate = voice(text)
    print(text); display(Audio(wave, rate=rate))